# EDA — RSNA Knee Abnormality Detection

Exploration détaillée des CSV officiels et des volumes NPZ locaux `(24, 224, 224)` uint8.

**Objectif du modèle :** 12 probabilités par *étude* (macro-AUC). Les labels sont au niveau study, pas au niveau série.

Les volumes restent dans `data/volumes.zip` (pas d'extraction).

In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import io
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": False})

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

CSV = ROOT / "data" / "csv"
VOLUMES = ROOT / "data" / "volumes.zip"

LABELS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]
PLANES = ["Sagittal", "Coronal", "Axial"]

train = pd.read_csv(CSV / "train.csv")
series = pd.read_csv(CSV / "train_series.csv")
test = pd.read_csv(CSV / "test.csv")
test_series = pd.read_csv(CSV / "test_series.csv")
sub = pd.read_csv(CSV / "sample_submission.csv")

labeled = train[train[LABELS].notna().all(axis=1)].copy()
unlabeled = train[train[LABELS].isna().all(axis=1)].copy()

z = zipfile.ZipFile(VOLUMES)


def load_vol(series_uid: str) -> np.ndarray:
    return np.load(io.BytesIO(z.read(f"{series_uid}.npz")))["data"]


print("train     ", train.shape)
print("series    ", series.shape)
print("test ex.  ", test.shape, test_series.shape)
print("volumes   ", VOLUMES.exists(), VOLUMES.stat().st_size / 1e9, "Go")


## 1. Inventaire

| Fichier | Rôle |
|---|---|
| `train.csv` | 1 ligne / étude : rapport + 12 labels (souvent vides) |
| `train_series.csv` | 1 ligne / série MRI : plan, fluid, fat-sat |
| `volumes.zip` | 1 `.npz` / série, clé `data` |
| `test.csv` / `test_series.csv` | **3 études d'exemple** — le vrai test (~1300) n'existe qu'au scoring |

In [ ]:
inv = pd.DataFrame({
    "fichier": ["train.csv", "train_series.csv", "volumes.zip", "test.csv", "test_series.csv"],
    "n": [
        len(train),
        len(series),
        sum(n.endswith(".npz") for n in z.namelist()),
        len(test),
        len(test_series),
    ],
})
display(inv)

print("études train sans série :", len(set(train.StudyInstanceUID) - set(series.StudyInstanceUID)))
print("séries orphelines       :", len(set(series.StudyInstanceUID) - set(train.StudyInstanceUID)))
print("SeriesInstanceUID unique:", series.SeriesInstanceUID.is_unique)
print("colonnes submission     :", list(sub.columns))


## 2. Le trou des labels — point le plus important

L'overview Kaggle le dit : *only a small subset of training studies carry per-condition labels*.
Les autres études n'ont que le **rapport texte**, souvent multilingue.

Conséquence : un modèle supervisé naïf n'a que quelques dizaines d'études. Le vrai levier, c'est d'extraire des labels (ou du signal) depuis `Report`.

In [ ]:
n_present = train[LABELS].notna().sum(axis=1)
print(n_present.value_counts().sort_index().rename("n_études"))
print()
print(f"labelisées (12/12) : {len(labeled):>5}  ({100 * len(labeled) / len(train):.2f}%)")
print(f"sans aucun label   : {len(unlabeled):>5}  ({100 * len(unlabeled) / len(train):.2f}%)")
print("labels partiels    :", int(((n_present > 0) & (n_present < 12)).sum()))
print()
print("missing rate identique sur les 12 colonnes :")
print(train[LABELS].isna().mean().round(4).to_string())

fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.bar(["sans label\n(rapports seuls)", "12 labels"], [len(unlabeled), len(labeled)], color=["#888", "#2a6"])
ax.set_ylabel("études")
ax.set_title("Couverture des labels (train)")
for i, v in enumerate([len(unlabeled), len(labeled)]):
    ax.text(i, v, f" {v}", va="bottom", ha="center")
plt.tight_layout()
plt.show()


## 3. Prévalence (n = 58)

Macro-AUC : les classes rares pèsent autant que les fréquentes. **MCL** et **Lateral OA** seront les plus fragiles.
Aucune étude labelisée n'est « tout négatif » — biais de sélection si on n'entraîne que sur ces 58.

In [ ]:
stats = pd.DataFrame({
    "n": labeled[LABELS].notna().sum(),
    "positifs": labeled[LABELS].sum().astype(int),
    "prevalence": labeled[LABELS].mean(),
})
stats["negatifs"] = stats["n"] - stats["positifs"]
stats = stats.sort_values("prevalence")
display(stats.round(3))

n_pos = labeled[LABELS].sum(axis=1)
print("\npositifs / étude labelisée")
print(n_pos.describe().round(2).to_string())
print("études 100% négatives :", int((n_pos == 0).sum()))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].barh(stats.index, stats["prevalence"], color="#c44")
axes[0].set_xlim(0, 1)
axes[0].set_xlabel("prévalence")
axes[0].set_title("Prévalence des 12 cibles (n=58)")
axes[1].hist(n_pos, bins=np.arange(0.5, 12.5, 1), color="#36c", edgecolor="white")
axes[1].set_xlabel("nb de cibles positives")
axes[1].set_ylabel("études")
axes[1].set_title("Densité multi-label")
plt.tight_layout()
plt.show()


## 4. Co-occurrence et corrélation

Des paires cliniques attendues (OA médiale ↔ ménisque médial, épanchement ↔ synovite) doivent apparaître.
Avec n=58, les corrélations sont **bruitées** — s'en servir comme prior, pas comme vérité.

In [ ]:
M = labeled[LABELS].astype(int)
cooc = M.T @ M
corr = M.corr()

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.4))
im0 = axes[0].imshow(cooc.values, cmap="Blues")
axes[0].set_xticks(range(12)); axes[0].set_xticklabels(LABELS, rotation=90, fontsize=8)
axes[0].set_yticks(range(12)); axes[0].set_yticklabels(LABELS, fontsize=8)
axes[0].set_title("Co-occurrence (nb d'études)")
plt.colorbar(im0, ax=axes[0], fraction=0.046)
im1 = axes[1].imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
axes[1].set_xticks(range(12)); axes[1].set_xticklabels(LABELS, rotation=90, fontsize=8)
axes[1].set_yticks(range(12)); axes[1].set_yticklabels(LABELS, fontsize=8)
axes[1].set_title("Corrélation de Pearson")
plt.colorbar(im1, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.show()

pairs = []
cols = list(LABELS)
for i in range(12):
    for j in range(i + 1, 12):
        pairs.append((corr.iloc[i, j], cols[i], cols[j], int(cooc.iloc[i, j])))
pairs.sort(reverse=True)
print("paires les plus corrélées")
for r, a, b, n in pairs[:8]:
    print(f"  {r:+.2f}   n={n:2d}   {a}  ×  {b}")
print("paires les moins corrélées")
for r, a, b, n in pairs[-5:]:
    print(f"  {r:+.2f}   n={n:2d}   {a}  ×  {b}")


## 5. Rapports radiologiques

Toujours présents, longueur très variable, **plusieurs langues**. Au test, le rapport **n'est pas fourni**.

Un modèle multimodal (image + texte) ne peut donc s'appuyer sur le texte **qu'à l'entraînement** (pseudo-labels, pretraining). L'inférence Kaggle est image-only.

In [ ]:
rep = train["Report"].fillna("")
print("rapports vides :", int((rep.str.strip() == "").sum()))
print(rep.str.len().describe().round(1).to_string())

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.hist(rep.str.len(), bins=40, color="#555")
ax.set_xlabel("longueur (caractères)")
ax.set_ylabel("études")
ax.set_title("Longueur des rapports")
plt.tight_layout()
plt.show()

LANG_KW = {
    "es": ["rodilla", "técnica", "tecnica", "rotura", "derrame", "menisco interno"],
    "en": ["findings", "impression", "knee mri", "meniscus tear", "acl reconstruction"],
    "nl": ["bevindingen", "klinische", "meniscusscheur", "knie rechts", "knie links"],
    "fr": ["genou", "compte rendu", "ménisque", "technique :", "irm du genou"],
    "de": ["befund", "meniskus", "knie mrt", "beurteilung"],
    "pt": ["joelho", "derrame articular", "ruptura"],
    "it": ["ginocchio", "referto", "lesione del"],
}


def guess_lang(text: str) -> str:
    t = text.lower()
    scores = {lang: sum(kw in t for kw in kws) for lang, kws in LANG_KW.items()}
    best, val = max(scores.items(), key=lambda kv: kv[1])
    return best if val else "other"


train = train.copy()
train["lang"] = rep.map(guess_lang)
print("\nlangue (heuristique mots-clés — approximatif)")
print(train["lang"].value_counts().to_string())

fig, ax = plt.subplots(figsize=(6, 3.2))
train["lang"].value_counts().plot.bar(ax=ax, color="#36c", rot=0)
ax.set_ylabel("études")
ax.set_title("Langue estimée des rapports")
plt.tight_layout()
plt.show()

print("\n--- extraits ---")
for lang in ["en", "es", "nl", "fr", "de", "other"]:
    hit = train[train["lang"] == lang]
    if hit.empty:
        continue
    txt = str(hit.iloc[0]["Report"]).replace("\n", " ")[:280]
    print(f"\n[{lang}] {txt}...")


In [ ]:
print("rapports : labelisées vs non labelisées")
cmp = pd.DataFrame({
    "labelisées": labeled["Report"].str.len().describe(),
    "non labelisées": unlabeled["Report"].str.len().describe(),
}).round(1)
display(cmp)

print("\nlangue × statut label")
train["is_labeled"] = train.StudyInstanceUID.isin(labeled.StudyInstanceUID)
display(pd.crosstab(train["lang"], train["is_labeled"], margins=True))


## 6. Protocoles MRI (séries)

Chaque étude a **les 3 plans**. Presque toutes ont plusieurs séries du même plan (T1 vs PD/T2, fat-sat, etc.).

Dans *ce* train, `Fluid_Sensitive` ≡ `Fat_Suppression` (corrélation 1.0). L'overview prévient que ce n'est **pas garanti** ailleurs — au test, traiter les deux flags séparément.

In [ ]:
print(series["Anatomical_Plane"].value_counts().to_string())
print()
print("Fluid_Sensitive == Fat_Suppression :", (series.Fluid_Sensitive == series.Fat_Suppression).mean())
display(pd.crosstab(series.Fluid_Sensitive, series.Fat_Suppression))
print()
print("plan × fluid-sensitive")
display(pd.crosstab(series.Anatomical_Plane, series.Fluid_Sensitive, margins=True))

per = series.groupby("StudyInstanceUID").size()
print("\nséries / étude")
print(per.describe().round(2).to_string())

n_planes = series.groupby("StudyInstanceUID")["Anatomical_Plane"].nunique()
print("études avec 3 plans :", int((n_planes == 3).sum()), "/", len(n_planes))

same_plane = series.groupby(["StudyInstanceUID", "Anatomical_Plane"]).size()
print("max de séries d'un même plan dans une étude :", int(same_plane.max()))
print("études avec doublon de plan :", int(same_plane.gt(1).groupby(level=0).any().sum()))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
per.hist(bins=range(3, int(per.max()) + 2), ax=axes[0], color="steelblue")
axes[0].set_title("séries / étude")
series["Anatomical_Plane"].value_counts().loc[PLANES].plot.bar(ax=axes[1], rot=0, color="steelblue")
axes[1].set_title("plan")
series["Fluid_Sensitive"].value_counts().sort_index().plot.bar(ax=axes[2], rot=0, color="steelblue")
axes[2].set_title("Fluid_Sensitive")
axes[2].set_xticklabels(["0 (T1-like)", "1 (T2/PD/STIR)"])
plt.tight_layout()
plt.show()


In [ ]:
proto = (
    series.groupby(["StudyInstanceUID", "Anatomical_Plane", "Fluid_Sensitive"])
    .size()
    .unstack("Fluid_Sensitive", fill_value=0)
)
proto.columns = ["n_non_fluid", "n_fluid"]
has_both = (proto["n_non_fluid"] > 0) & (proto["n_fluid"] > 0)
print("études ayant fluid ET non-fluid, par plan")
print(has_both.groupby("Anatomical_Plane").mean().reindex(PLANES).round(3).to_string())

combo = series.assign(proto=np.where(series.Fluid_Sensitive == 1, "fluid", "non-fluid"))
combo["key"] = combo["Anatomical_Plane"] + " / " + combo["proto"]
print("\ncombinaisons les plus fréquentes")
print(combo["key"].value_counts().to_string())


## 7. Volumes NPZ

Shape uniforme `(D=24, H=224, W=224)`, `uint8`. Les intensités ont déjà été normalisées par l'auteur du dataset.

On vérifie la couverture vis-à-vis de `train_series.csv`, puis un échantillon d'intensités par plan.

In [ ]:
npz_ids = {Path(n).stem for n in z.namelist() if n.endswith(".npz")}
series_ids = set(series.SeriesInstanceUID)
print(f"npz              : {len(npz_ids):,}")
print(f"séries train     : {len(series_ids):,}")
print(f"séries sans npz  : {len(series_ids - npz_ids):,}")
print(f"npz hors train   : {len(npz_ids - series_ids):,}")

sample_uid = series.SeriesInstanceUID.iloc[0]
vol = load_vol(sample_uid)
print(f"\nexemple shape={vol.shape} dtype={vol.dtype} min={vol.min()} max={vol.max()} mean={vol.mean():.1f}")


In [ ]:
rng = np.random.default_rng(0)
sample_n = 120
idx = rng.choice(len(series), size=sample_n, replace=False)
rows = series.iloc[idx]

records = []
for _, r in rows.iterrows():
    v = load_vol(r.SeriesInstanceUID)
    records.append({
        "plane": r.Anatomical_Plane,
        "fluid": int(r.Fluid_Sensitive),
        "mean": float(v.mean()),
        "std": float(v.std()),
        "p99": float(np.percentile(v, 99)),
        "nonzero": float((v > 5).mean()),
    })
inten = pd.DataFrame(records)
display(inten.groupby(["plane", "fluid"])[["mean", "std", "nonzero"]].mean().round(2))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for plane, g in inten.groupby("plane"):
    axes[0].hist(g["mean"], bins=20, alpha=0.45, label=plane)
axes[0].set_title("intensité moyenne (échantillon)")
axes[0].legend()
inten.boxplot(column="mean", by="fluid", ax=axes[1])
axes[1].set_title("mean vs Fluid_Sensitive")
axes[1].set_xlabel("fluid")
plt.suptitle("")
plt.tight_layout()
plt.show()


## 8. Galerie — une étude, 3 plans

Coupes `z = 0, 6, 12, 17, 23` (volume déjà rééchantillonné à 24 slices).

In [ ]:
def show_study(study_uid: str, title: str = ""):
    rows = series[series.StudyInstanceUID == study_uid]
    zs = [0, 6, 12, 17, 23]
    fig, axes = plt.subplots(len(PLANES), len(zs), figsize=(11, 6.6))
    for i, plane in enumerate(PLANES):
        hit = rows[rows.Anatomical_Plane == plane]
        if hit.empty:
            for ax in axes[i]:
                ax.axis("off")
            continue
        # préfère une série fluid-sensitive si disponible
        hit = hit.sort_values("Fluid_Sensitive", ascending=False)
        v = load_vol(hit.SeriesInstanceUID.iloc[0])
        fs = int(hit.Fluid_Sensitive.iloc[0])
        for ax, j in zip(axes[i], zs):
            ax.imshow(v[j], cmap="gray", vmin=0, vmax=255)
            ax.set_xticks([]); ax.set_yticks([])
            if i == 0:
                ax.set_title(f"z={j}")
        axes[i][0].set_ylabel(f"{plane}\nfluid={fs}", rotation=0, ha="right", va="center")
    fig.suptitle(title or study_uid[-24:], fontsize=11)
    plt.tight_layout()
    plt.show()


study = labeled.StudyInstanceUID.iloc[0]
row = labeled.set_index("StudyInstanceUID").loc[study]
pos = [c for c in LABELS if row[c] == 1]
print("étude :", study)
print("positifs :", pos)
print("rapport :")
print(str(row.Report)[:500])
show_study(study, title="étude labelisée — 3 plans (série fluid si possible)")


## 9. Galerie par pathologie

Pour chaque cible : 1 étude positive, coupe sagittale centrale (série fluid-sensitive si possible).
n=58 seulement → ces images illustrent, elles ne « prouvent » pas le signe radiologique.

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
axes = axes.ravel()
for ax, lab in zip(axes, LABELS):
    hits = labeled[labeled[lab] == 1]
    if hits.empty:
        ax.axis("off")
        continue
    sid = hits.StudyInstanceUID.iloc[0]
    cand = series[(series.StudyInstanceUID == sid) & (series.Anatomical_Plane == "Sagittal")]
    cand = cand.sort_values("Fluid_Sensitive", ascending=False)
    v = load_vol(cand.SeriesInstanceUID.iloc[0])
    ax.imshow(v[12], cmap="gray", vmin=0, vmax=255)
    ax.set_title(lab, fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Sagittal z=12 — 1 positif / cible", fontsize=12)
plt.tight_layout()
plt.show()


## 10. Fluid vs non-fluid (même étude, même plan)

Les séquences fluid-sensitive (T2 / PD / STIR) mettent en évidence épanchement, kyste, contusion.
Comparer les deux protocoles sur la même anatomie.

In [ ]:
pairs = (
    series.groupby(["StudyInstanceUID", "Anatomical_Plane"])
    .filter(lambda g: set(g.Fluid_Sensitive) == {0, 1})
)
pair_studies = pairs.StudyInstanceUID.unique()
print("études avec fluid ET non-fluid sur au moins un plan :", len(pair_studies))

cand = labeled.StudyInstanceUID[labeled.StudyInstanceUID.isin(pair_studies)]
sid = cand.iloc[0] if len(cand) else pair_studies[0]
rows = series[series.StudyInstanceUID == sid]
print("étude", sid[-20:])

fig, axes = plt.subplots(3, 4, figsize=(11, 8))
for i, plane in enumerate(PLANES):
    subp = rows[rows.Anatomical_Plane == plane]
    for j, fluid in enumerate([0, 1]):
        hit = subp[subp.Fluid_Sensitive == fluid]
        ax0 = axes[i, 2 * j]
        ax1 = axes[i, 2 * j + 1]
        if hit.empty:
            ax0.axis("off"); ax1.axis("off")
            continue
        v = load_vol(hit.SeriesInstanceUID.iloc[0])
        ax0.imshow(v[6], cmap="gray", vmin=0, vmax=255)
        ax1.imshow(v[12], cmap="gray", vmin=0, vmax=255)
        ax0.set_xticks([]); ax0.set_yticks([])
        ax1.set_xticks([]); ax1.set_yticks([])
        if i == 0:
            ax0.set_title(f"fluid={fluid} z=6")
            ax1.set_title(f"fluid={fluid} z=12")
        if j == 0:
            ax0.set_ylabel(plane, rotation=0, ha="right", va="center")
plt.suptitle("Même étude — fluid vs non-fluid")
plt.tight_layout()
plt.show()


## 11. Test exemple vs train

Le test public/privé réel n'est pas là. Ces 3 études servent uniquement à valider le format du notebook de submit.
Pas de `Report` au test.

In [ ]:
print("test études :", len(test), "  séries :", len(test_series))
print("report dans test.csv ?", "Report" in test.columns)
display(test_series.groupby("StudyInstanceUID").agg(
    n_series=("SeriesInstanceUID", "count"),
    planes=("Anatomical_Plane", lambda s: ",".join(sorted(s.unique()))),
    n_fluid=("Fluid_Sensitive", "sum"),
))
print("submission colonnes == labels + uid :", list(sub.columns) == ["StudyInstanceUID"] + LABELS)


## 12. Implications pour le modèle

1. **58 labels officiels** : trop peu pour un CNN 3D seul. Priorité = pseudo-labels depuis les rapports (LLM / règles / modèle texte), puis entraînement image sur ~4407 études.
2. **Inférence image-only** : tout ce qui vient du texte doit être distillé dans le modèle vision.
3. **Macro-AUC** : sur-pondérer MCL, Lateral OA, Baker (rares dans le petit set labelisé — à re-mesurer après pseudo-labels).
4. **Entrée modèle** : chaque étude a 3 plans et souvent 5–6 séries. Un baseline raisonnable : 1 volume sagittal fluid + 1 coronal + 1 axial, ou pooling / MIL sur les séries.
5. **NPZ 224×24** : suffisant pour un baseline local. Limite probable sur ménisque / cartilage / petites fractures.
6. **Fluid ≡ fat-sat dans le train** : ne pas hardcoder cette égalité pour le test.
7. **Biais des 58** : zéro étude « tout négatif ». Un modèle entraîné seulement là-dessus prédira trop de positifs.

Prochaine étape naturelle : extraire des labels faibles depuis `Report`, mesurer l'accord avec les 58 études gold, puis un DataLoader 2.5D sur les NPZ.